In [ ]:
# Local roots: copy 05_GitHub/.env.example to .env (gitignored).
import os
from pathlib import Path

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    env = _p / ".env"
    if env.exists():
        for line in env.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
        break

CALLING = os.environ.get("SUNFLOWER_CALLING", str(Path.cwd()))
CHOCALLATE = os.environ["CHOCALLATE"] if "CHOCALLATE" in os.environ else ""
REF = os.environ.get("SUNFLOWER_REF", "")


# Sunflower calling pipeline (technical replicates)

**Phase 1 (below):** first ChoCallate run → filters → `snp_het_LD.vcf` for replicate QC.

**Phase 2:** IBS → BAM merge → recall → filters on `raw_vcf_merged`.

> Run the `raw_vcf_merged` cells only after the recall on merged BAMs has finished.


In [ ]:
!python make_samples_tsv.py \
raw_reads \
samples.tsv


In [ ]:
!REF="${REF}";TMP_REF=$(mktemp --suffix=.fna); gzip -dc "$REF" > "$TMP_REF"; samtools faidx "$TMP_REF"; cp "${TMP_REF}.fai" "$(basename "${REF%.gz}").fai"; rm -f "$TMP_REF" "${TMP_REF}.fai"


In [ ]:
!awk 'BEGIN{OFS="\t"} {print $1, 0, $2}' \
HanXRQr2.0-SUNRISE.fna.fai > HanXRQr2.0-SUNRISE.bed


In [ ]:
!nextflow run ${CHOCALLATE}/main.nf \
-params-file config.yaml


In [ ]:
!bgzip raw_vcf/per_sample/*


In [ ]:
!for vcf in raw_vcf/per_sample/*.vcf.gz; do tabix -p vcf "$vcf"; done


In [ ]:
!bcftools merge \
    -m all \
    -Oz \
    -o raw_vcf/all_samples.merged.vcf.gz \
    raw_vcf/per_sample/*.vcf.gz


In [ ]:
!bcftools stats raw_vcf/all_samples.merged.vcf.gz | grep '^SN'


In [ ]:
!bgzip raw_vcf_merged/all_samples.merged_min_2.vcf


In [ ]:
!tabix raw_vcf_merged/all_samples.merged_min_2.vcf.gz


In [ ]:
!bcftools merge \
    -m all \
    -Oz \
    -o raw_vcf/all_samples.merged_1549.vcf.gz \
    raw_vcf_merged/all_samples.merged_min_2.vcf.gz raw_vcf/per_sample/15*.vcf.gz raw_vcf/per_sample/49*.vcf.gz


In [ ]:
!gunzip raw_vcf/all_samples.merged_1549.vcf.gz


In [ ]:
%%bash

CPU=30

vcftools \
--vcf raw_vcf/all_samples.merged_1549.vcf \
--remove-filtered-all \
--min-alleles 2 \
--recode \
--recode-INFO-all \
--stdout | bcftools +fixploidy --threads $CPU -Oz \
-o raw_vcf/all_samples.merged_1549_min_2.vcf


In [ ]:
!bcftools stats raw_vcf/all_samples.merged_1549_min_2.vcf | grep '^SN'


In [ ]:
!bcftools merge \
    -m all \
    -Oz \
    -o raw_vcf/per_sample/outputs/merged_vcf/all_samples.merged.vcf.gz \
    raw_vcf/per_sample/outputs/merged_vcf/*.vcf.gz


In [ ]:
!gunzip raw_vcf/per_sample/outputs/merged_vcf/all_samples.merged.vcf.gz


In [ ]:
!sed -i 's/^##fileformat=VCFv4.3/##fileformat=VCFv4.2/' raw_vcf/per_sample/outputs/merged_vcf/all_samples.merged.vcf


In [ ]:
%%bash

CPU=30

vcftools \
--vcf raw_vcf/per_sample/outputs/merged_vcf/all_samples.merged.vcf \
--remove-filtered-all \
--min-alleles 2 \
--recode \
--recode-INFO-all \
--stdout | bcftools +fixploidy --threads $CPU -Oz \
-o raw_vcf/per_sample/outputs/merged_vcf/all_samples.merged_min_2.vcf


In [ ]:
!bcftools stats raw_vcf/per_sample/outputs/merged_vcf/all_samples.merged_min_2.vcf | grep '^SN'


In [ ]:
!bcftools stats raw_vcf/per_sample/outputs/merged_vcf/29.vcf.gz | grep '^SN'


In [ ]:
!bcftools stats raw_vcf_merged/per_sample/29.0.vcf.gz | grep '^SN'


In [ ]:
!gunzip raw_vcf/all_samples.merged.vcf.gz


In [ ]:
!sed -i 's/^##fileformat=VCFv4.3/##fileformat=VCFv4.2/' raw_vcf/all_samples.merged.vcf


In [ ]:
%%bash

CPU=30

vcftools \
--vcf raw_vcf/all_samples.merged.vcf \
--remove-filtered-all \
--min-alleles 2 \
--max-alleles 2 \
--recode \
--recode-INFO-all \
--stdout | bcftools +fixploidy --threads $CPU -Oz -o raw_vcf/all_samples.merged_min_max_2.vcf


In [ ]:
!vcftools --vcf raw_vcf/all_samples.merged_min_max_2.vcf --missing-site --out raw_vcf/missrate_raw


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.genfromtxt("raw_vcf/missrate_raw.lmiss", delimiter="\t", skip_header=1, usecols=-1)
plt.hist(data, bins=10, edgecolor="black")
plt.ylabel("Number of variants")
plt.xlabel("Missing-data fraction per site")
plt.axvline(x=0.1, color="red", linestyle="--", linewidth=2)
plt.show()


In [ ]:
!plink2 --vcf raw_vcf/all_samples.merged_min_max_2.vcf --out filt_vcf/filt_geno \
--allow-extra-chr --geno 0.4 --export vcf


In [ ]:
!plink2 --vcf  filt_vcf/filt_geno.vcf --allow-extra-chr --maf 0.1 \
--export vcf --out  filt_vcf/filt_maf


In [ ]:
!sed -i 's/^##fileformat=VCFv4.3/##fileformat=VCFv4.2/' filt_vcf/filt_maf.vcf


In [ ]:
!vcftools --vcf filt_vcf/filt_maf.vcf --missing-indv \
--out filt_vcf/missrate_mind


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.genfromtxt("filt_vcf/missrate_mind.imiss", delimiter="\t", skip_header=1, usecols=-1)
plt.hist(data, bins=100, edgecolor="black")
plt.xlabel("Missing-site fraction per sample")
plt.ylabel("Number of samples")
plt.axvline(x=0.05, color="red", linestyle="--", linewidth=2)
plt.show()


In [ ]:
!plink2 --vcf filt_vcf/filt_maf.vcf --out filt_vcf/filt_mind \
--allow-extra-chr --mind 0.8 --export vcf


In [ ]:
!bcftools annotate --set-id +'%CHROM\_%POS\_%REF\_%ALT' filt_vcf/filt_mind.vcf \
-o filt_vcf/filt_geno_id.vcf


In [ ]:
!conda install beagle -y


In [ ]:
!beagle \
  gt=filt_vcf/filt_geno_id.vcf \
  out=filt_vcf/filt_mind_imputed \
  nthreads=20 ne=1000 && \
gunzip filt_vcf/filt_mind_imputed.vcf.gz -f 1> log_beagle


In [ ]:
!plink2 --vcf filt_vcf/filt_mind_imputed.vcf --allow-extra-chr \
--geno-counts --out filt_vcf/snps_geno_cnts


In [ ]:
!awk -v 'FS=\t' '$0 !~ /#/ && ($6 / ($5 + $6 + $7) <= 0.05) && ($5 > 0) && ($7 > 0) {print $2}' \
filt_vcf/snps_geno_cnts.gcount > filt_vcf/lowhet_snp_id.txt
!plink2 --vcf filt_vcf/filt_mind_imputed.vcf --allow-extra-chr --extract filt_vcf/lowhet_snp_id.txt \
--export vcf --out filt_vcf/lowhet_geno_filtered


In [ ]:
!plink2 --vcf filt_vcf/lowhet_geno_filtered.vcf --bad-ld --allow-extra-chr \
--indep-pairwise 500k 0.2 --out filt_vcf/snps_prune


In [ ]:
!plink2 --vcf filt_vcf/lowhet_geno_filtered.vcf --allow-extra-chr \
--extract filt_vcf/snps_prune.prune.in --export vcf --out filt_vcf/snp_het_LD


In [ ]:
!grep "#CHROM" filt_vcf/snp_het_LD.vcf


## Technical replicates: IBS on `snp_het_LD.vcf`

A name such as `12-1` / `12-4` is two technical replicates of **one** line (`12`).

- Metric: `plink2 --make-king-table` → `.kin0`, **IBS = 1 − IBS0** (PLINK 1 `--genome` PI_HAT analogue).
- Threshold: **IBS ≥ 0.85** (below this the pair is too distant; see `replicate_warnings.txt`).
- Written: `filt_vcf/replicate_ibs_pairs.tsv`, `replicate_merge_plan.tsv`.

> Run after `filt_vcf/snp_het_LD.vcf` exists.


In [ ]:
import re
import itertools
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

VCF = Path("filt_vcf/snp_het_LD.vcf")
IBS_PREFIX = Path("filt_vcf/replicate_ibs")
KIN0 = Path(f"{IBS_PREFIX}.kin0")
IBS_THRESHOLD = 0.85

# PLINK 2 --make-king-table. Similarity: IBS = 1 - IBS0
# (share of loci without opposite homozygotes; analogue of PLINK 1 PI_HAT).
!plink2 --vcf {VCF} --double-id --allow-extra-chr --make-king-table --out {IBS_PREFIX}

if not KIN0.exists():
    raise FileNotFoundError(f"{KIN0} is missing — check {IBS_PREFIX}.log")

hdr = !grep "^#CHROM" {VCF}
samples = hdr[0].strip().split("\t")[9:]

def parse_line_replicate(name: str):
    m = re.match(r"^(\d+)-(\d+)$", name)
    if not m:
        return None, None
    return m.group(1), m.group(2)

line_to_reps = defaultdict(list)
for s in samples:
    line, rep = parse_line_replicate(s)
    if line is not None:
        line_to_reps[line].append(s)

kin = pd.read_csv(KIN0, sep="\t")
kin["s1"] = kin["IID1"].astype(str)
kin["s2"] = kin["IID2"].astype(str)
kin["ibs"] = 1.0 - kin["IBS0"].astype(float)

ibs = {}
for _, row in kin.iterrows():
    ibs[(row["s1"], row["s2"])] = row["ibs"]
    ibs[(row["s2"], row["s1"])] = row["ibs"]

def get_ibs(a, b):
    if a == b:
        return 1.0
    return ibs.get((a, b), np.nan)

pair_rows = []
warn_lines = []
for line in sorted(line_to_reps, key=int):
    reps = sorted(line_to_reps[line], key=lambda x: (int(parse_line_replicate(x)[1]), x))
    if len(reps) < 2:
        continue
    for a, b in itertools.combinations(reps, 2):
        v = get_ibs(a, b)
        ok = (not np.isnan(v)) and v >= IBS_THRESHOLD
        pair_rows.append({
            "line": line, "rep_a": a, "rep_b": b, "ibs": v,
            "ibs0": 1.0 - v if not np.isnan(v) else np.nan,
            "ok_for_merge": ok,
        })
        if not ok:
            warn_lines.append(
                f"WARNING: line {line}: {a} vs {b} IBS={v:.4f} < {IBS_THRESHOLD}"
            )

pairs_df = pd.DataFrame(pair_rows)
pairs_df.to_csv("filt_vcf/replicate_ibs_pairs.tsv", sep="\t", index=False)
with open("filt_vcf/replicate_warnings.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(warn_lines) + "\n" if warn_lines else "All pairs IBS >= threshold.\n")

print(f"pairs: {len(pairs_df)}, warnings: {len(warn_lines)}")
for w in warn_lines[:15]:
    print(w)


In [ ]:
def select_reps_for_line(reps):
    """Drop a replicate if IBS against any other kept replicate is below the threshold."""
    reps = sorted(reps, key=lambda x: (int(parse_line_replicate(x)[1]), x))
    if len(reps) == 1:
        return reps, []
    if len(reps) == 2:
        a, b = reps
        v = get_ibs(a, b)
        if np.isnan(v) or v < IBS_THRESHOLD:
            return [], reps
        return reps, []

    kept = list(reps)
    while True:
        drop = []
        for r in kept:
            others = [x for x in kept if x != r]
            if any(get_ibs(r, o) < IBS_THRESHOLD for o in others):
                drop.append(r)
        if not drop:
            break
        kept = [r for r in kept if r not in drop]
    dropped = [r for r in reps if r not in kept]
    return (kept, dropped) if kept else ([], reps)

plan_rows = []
for line in sorted(line_to_reps, key=int):
    reps = line_to_reps[line]
    if len(reps) < 2:
        plan_rows.append({
            "line": line, "action": "single",
            "use_reps": ",".join(reps), "drop_reps": "", "merged_id": reps[0],
        })
        continue
    use, drop = select_reps_for_line(reps)
    if not use:
        plan_rows.append({
            "line": line, "action": "exclude",
            "use_reps": "", "drop_reps": ",".join(reps), "merged_id": "",
        })
    elif len(use) == 1:
        plan_rows.append({
            "line": line, "action": "keep_one",
            "use_reps": use[0], "drop_reps": ",".join(drop), "merged_id": use[0],
        })
    else:
        plan_rows.append({
            "line": line, "action": "merge_bam",
            "use_reps": ",".join(use), "drop_reps": ",".join(drop), "merged_id": line,
        })

plan_df = pd.DataFrame(plan_rows)
plan_df.to_csv("filt_vcf/replicate_merge_plan.tsv", sep="\t", index=False)
print(plan_df["action"].value_counts())
plan_df[plan_df["action"].isin(["exclude", "merge_bam"])]


In [ ]:
!nextflow run ${CHOCALLATE}/main.nf -params-file config_align_bam.yaml


## BAM: manifest and replicate merge

Create `filt_vcf/bam_manifest.tsv` (`sample_id<TAB>path_to.bam`).

Helper (from the calling root):

```bash
python collect_bam_manifest.py work filt_vcf/bam_manifest.tsv LEFT_ALIGN_INDELS
```

Stage: `LEFT_ALIGN_INDELS` (or `FILTER_MAPPING_BAM` if left-align is off).


In [ ]:
from pathlib import Path

BAM_MANIFEST_TSV = Path("filt_vcf/bam_manifest.tsv")
if not BAM_MANIFEST_TSV.exists():
    raise FileNotFoundError(
        "Create filt_vcf/bam_manifest.tsv by hand or by searching work/\n"
        "Example: find work -path '*LEFT_ALIGN_INDELS*' -name output.bam"
    )

bam_map = {}
for line in BAM_MANIFEST_TSV.read_text(encoding="utf-8").splitlines():
    if not line.strip() or line.startswith("#"):
        continue
    sid, bpath = line.split("\t", 1)
    bam_map[sid.strip()] = Path(bpath.strip())

print(f"BAMs in the manifest: {len(bam_map)}")


In [ ]:
import subprocess
from pathlib import Path
import pandas as pd

MERGED_BAM_DIR = Path("merged_bam")
MERGED_BAM_DIR.mkdir(exist_ok=True)

plan_df = pd.read_csv("filt_vcf/replicate_merge_plan.tsv", sep="\t")
merge_log = []

for _, row in plan_df.iterrows():
    action = row["action"]
    mid = str(row["merged_id"]).strip()

    if action == "exclude":
        merge_log.append(f"SKIP line {row['line']}: excluded")
        continue

    if action in ("single", "keep_one"):
        rep = str(row["use_reps"]).split(",")[0].strip()
        src = bam_map[rep]
        dst = MERGED_BAM_DIR / f"{mid}.bam"
        if dst.exists():
            dst.unlink()
        dst.symlink_to(src.resolve())
        subprocess.run(["samtools", "index", "-@", "8", str(dst)], check=True)
        merge_log.append(f"LINK {rep} -> {dst}")
        continue

    if action == "merge_bam":
        reps = [r.strip() for r in str(row["use_reps"]).split(",") if r.strip()]
        bams = [str(bam_map[r].resolve()) for r in reps]
        out_bam = MERGED_BAM_DIR / f"{mid}.bam"
        tmp = MERGED_BAM_DIR / f"{mid}.merge_tmp.bam"
        subprocess.run(["samtools", "merge", "-f", "-o", str(tmp), *bams], check=True)
        subprocess.run([
            "samtools", "addreplacerg",
            "-r", f"@RG\\tID:{mid}\\tSM:{mid}\\tPL:ILLUMINA",
            "-o", str(out_bam), str(tmp),
        ], check=True)
        tmp.unlink(missing_ok=True)
        subprocess.run(["samtools", "index", "-@", "8", str(out_bam)], check=True)
        merge_log.append(f"MERGE {reps} -> {out_bam}")

Path("filt_vcf/bam_merge.log").write_text("\n".join(merge_log) + "\n", encoding="utf-8")
print("\n".join(merge_log))


In [ ]:
rows = []
for bam in sorted(Path("merged_bam").glob("*.bam")):
    rows.append(f"{bam.stem}\t{bam.resolve()}\n")
Path("samples_merged.tsv").write_text("".join(rows), encoding="utf-8")
print(f"samples: {len(rows)}")


## Recall (ChoCallate, `format: bam`)


In [ ]:
merged_config = """input:
  samples_tsv: "${CALLING}/samples_merged.tsv"
  reference_genome: "${REF}"
  reference_index_dir: "${REF}"
  include_bed: "${CALLING}/HanXRQr2.0-SUNRISE.bed"
  exclude_bed: null
  format: "bam"
  reads_type: "pe"

output:
  directory: "${CALLING}/raw_vcf_merged"
  type: "sample"
  format: "vcf"

ref_genome:
  bgzip: true

mapping:
  mapper: "bowtie2"
  extra_args: ""
  cpu: 15

bam_filter:
  min_map_qual: 10

rmdup:
  enabled: false

left_align_indels:
  enabled: true

coverage:
  min_coverage: 2

calling:
  callers: "bcftools,gatk,freebayes"
  min_snp_qual: 20
  cpu: 4
  bcftools:
    call:
      extra_args: ""
    mpileup:
      extra_args: ""
  freebayes:
    extra_args: ""
  gatk:
    extra_args: ""

consensus:
  threshold: 2
  cpu: 4
"""
Path("config_merged.yaml").write_text(merged_config, encoding="utf-8")
print("config_merged.yaml written. Run:")
print("  nextflow run ${CHOCALLATE}/main.nf -params-file config_merged.yaml")


In [ ]:
# !nextflow run ${CHOCALLATE}/main.nf -params-file config_merged.yaml


## Phase 2: merge and filter VCFs (`raw_vcf_merged`)


In [ ]:
!for vcf in raw_vcf_merged/per_sample/*.vcf.gz; do tabix -p vcf "$vcf"; done


In [ ]:
!bcftools merge \
    -m all \
    -Oz \
    -o raw_vcf_merged/all_samples.merged.vcf.gz \
    raw_vcf_merged/per_sample/*.vcf.gz


In [ ]:
!gunzip -f raw_vcf_merged/all_samples.merged.vcf.gz


In [ ]:
!sed -i 's/^##fileformat=VCFv4.3/##fileformat=VCFv4.2/' raw_vcf_merged/all_samples.merged.vcf


In [ ]:
%%bash
CPU=30
vcftools \
--vcf raw_vcf_merged/all_samples.merged.vcf \
--remove-filtered-all \
--min-alleles 2 \
--max-alleles 2 \
--recode \
--recode-INFO-all \
--stdout | bcftools +fixploidy --threads $CPU -Oz -o raw_vcf_merged/all_samples.merged_min_max_2.vcf


In [ ]:
!mkdir -p filt_vcf_merged
!plink2 --vcf raw_vcf_merged/all_samples.merged_min_max_2.vcf --out filt_vcf_merged/filt_geno \
--allow-extra-chr --geno 0.4 --export vcf


In [ ]:
!plink2 --vcf filt_vcf_merged/filt_geno.vcf --allow-extra-chr --maf 0.1 \
--export vcf --out filt_vcf_merged/filt_maf


In [ ]:
!plink2 --vcf filt_vcf_merged/filt_maf.vcf --out filt_vcf_merged/filt_mind \
--allow-extra-chr --mind 0.8 --export vcf


In [ ]:
!bcftools annotate --set-id +'%CHROM\_%POS\_%REF\_%ALT' filt_vcf_merged/filt_mind.vcf \
-o filt_vcf_merged/filt_geno_id.vcf


In [ ]:
!beagle \
  gt=filt_vcf_merged/filt_geno_id.vcf \
  out=filt_vcf_merged/filt_mind_imputed \
  nthreads=20 ne=1000 && \
gunzip filt_vcf_merged/filt_mind_imputed.vcf.gz -f


In [ ]:
!plink2 --vcf filt_vcf_merged/filt_mind_imputed.vcf --allow-extra-chr \
--geno-counts --out filt_vcf_merged/snps_geno_cnts


In [ ]:
!awk -v 'FS=\t' '$0 !~ /#/ && ($6 / ($5 + $6 + $7) <= 0.05) && ($5 > 0) && ($7 > 0) {print $2}' \
filt_vcf_merged/snps_geno_cnts.gcount > filt_vcf_merged/lowhet_snp_id.txt
!plink2 --vcf filt_vcf_merged/filt_mind_imputed.vcf --allow-extra-chr --extract filt_vcf_merged/lowhet_snp_id.txt \
--export vcf --out filt_vcf_merged/lowhet_geno_filtered


In [ ]:
!plink2 --vcf filt_vcf_merged/lowhet_geno_filtered.vcf --bad-ld --allow-extra-chr \
--indep-pairwise 500k 0.2 --out filt_vcf_merged/snps_prune


In [ ]:
!plink2 --vcf filt_vcf_merged/lowhet_geno_filtered.vcf --allow-extra-chr \
--extract filt_vcf_merged/snps_prune.prune.in --export vcf --out filt_vcf_merged/snp_het_LD


In [ ]:
!grep "#CHROM" filt_vcf_merged/snp_het_LD.vcf
